# Disease + Spaceflight Module Curation Pipeline
**OMIM · GWAS · Spaceflight Module · PCNet 2.2 → pkl conversion**

Compiles disease-gene associations from OMIM and GWAS Catalog, harmonizes
nomenclature via MONDO, appends a spaceflight query module from three missions
(JAXA CFE, NASA Twins, Inspiration4), and converts the PCNet 2.2 interactome
from `.cx` to `.pkl`.

**Pipeline Steps:**
- Step 0 — Install packages, set up directories, download MONDO
- Step 1 — Convert PCNet 2.2 `.cx` → `.pkl` interactome
- Step 2 — Load MONDO and parse cross-reference tables
- Step 3 — Verify lookup dictionaries
- Step 4 — Map OMIM entries to MONDO
- Step 5 — Map GWAS Catalog entries to MONDO
- Step 6 — Combine, filter, and save disease gene sets
- Step 7 — Build spaceflight module and inject into `gene_modules.pkl`

**Expected input files (place before running):**
```
data/
  input/
    disease/
      genemap2.txt                     ← omim.org/downloads
      gwas_catalog_associations.tsv    ← ebi.ac.uk/gwas
    spaceflight/
      Supplementary_Data_1.xlsx        ← Ohshima et al. 2022 (JAXA CFE)
      aau8650_table_s2.xlsx            ← Garrett-Bakelman et al. 2019 (NASA Twins)
      Supplementary_Data_2.xlsx        ← Malkani et al. 2023 (Inspiration4)
    ppi/
      PCNet_2.2.cx                     ← NDEx / PCNet download
```

**Generated outputs:**
```
data/
  processed/
    ppi/
      ppi_network.pkl                  ← converted interactome (Step 1)
    disease/
      mondo_xref_table.csv             ← MONDO cross-reference table
    gene_modules/
      gene_modules.pkl                 ← disease + spaceflight gene sets
  output/
    csv/
      disease_gene_summary.csv
      disease_gene_mapping_audit.csv
      failed_mappings_for_review.csv
      removed_diseases.csv
      spaceflight_gene_audit.csv
```


## Step 0 — Install Packages, Set Up Directories, Download MONDO

In [ ]:
# Dependencies are pinned in requirements.txt / environment.yml.
# Set up your environment before running this notebook:
#
#   conda env create -f environment.yml
#   conda activate spaceflight-netsep
#
# Or with pip:
#   pip install -r requirements.txt
#
# Reference (do not run here — use the commands above instead):
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])
print("Environment ready — proceeding with notebook. ✅")

In [1]:
import utils  # shared helpers (load, save, disease_to_filename, gene mapping)

import urllib.request
import os

# ── Directory layout ──────────────────────────────────────────────────────────
DIRS = [
    "data/input/disease",
    "data/input/gene_info",
    "data/input/spaceflight",
    "data/input/ppi",
    "data/processed/ppi",
    "data/processed/disease",
    "data/processed/gene_modules",
    "data/output",
    "data/output/csv",
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print("Directories created.")

# ── Download MONDO ontology ───────────────────────────────────────────────────
MONDO_URL  = "https://github.com/monarch-initiative/mondo/releases/latest/download/mondo.obo"
MONDO_PATH = "data/input/disease/mondo.obo"

if not os.path.exists(MONDO_PATH):
    print("Downloading MONDO ontology...")
    urllib.request.urlretrieve(MONDO_URL, MONDO_PATH)
    print("MONDO downloaded successfully.")
else:
    print(f"MONDO already present at {MONDO_PATH} — skipping download.")

# ── Input-file checklist ──────────────────────────────────────────────────────
required_inputs = [
    "data/input/disease/mondo.obo",
    "data/input/disease/genemap2.txt",
    "data/input/disease/gwas_catalog_associations.tsv",
    "data/input/spaceflight/Supplementary_Data_1.xlsx",
    "data/input/spaceflight/aau8650_table_s2.xlsx",
    "data/input/spaceflight/Supplementary_Data_2.xlsx",
    "data/input/ppi/PCNet_2.2.cx",
    utils.GENE_INFO_PATH,   # NCBI Homo_sapiens.gene_info (see utils.py)
]

print("\nInput file checklist:")
all_present = True
for f in required_inputs:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    if "MISSING" in status:
        all_present = False
    print(f"  {status}  {f}")

if not all_present:
    print("\n⚠️  Some input files are missing. Place them before continuing.")
else:
    print("\n✅ All input files present.")


Directories created.
MONDO already present at data/input/disease/mondo.obo — skipping download.

Input file checklist:
  ✅  data/input/disease/mondo.obo
  ✅  data/input/disease/genemap2.txt
  ✅  data/input/disease/gwas_catalog_associations.tsv
  ✅  data/input/spaceflight/Supplementary_Data_1.xlsx
  ✅  data/input/spaceflight/aau8650_table_s2.xlsx
  ✅  data/input/spaceflight/Supplementary_Data_2.xlsx
  ✅  data/input/ppi/PCNet_2.2.cx
  ✅  data/input/gene_info/Homo_sapiens.gene_info

✅ All input files present.


## Step 1 — Convert PCNet 2.2 `.cx` → `.pkl` Interactome

Reads `data/input/ppi/PCNet_2.2.cx`, parses nodes (Entrez ID + gene symbol)
and edges, removes self-loops, retains only the largest connected component,
and serialises the result as a NetworkX `Graph` to
`data/processed/ppi/ppi_network.pkl`.

**Output:** `data/processed/ppi/ppi_network.pkl`

> **Skip this step** if `ppi_network.pkl` already exists — the cell detects
> the cached file and loads it instead of reprocessing.


In [2]:
import json
import pickle
import networkx as nx

CX_PATH  = "data/input/ppi/PCNet_2.2.cx"
PKL_PATH = "data/processed/ppi/ppi_network.pkl"

if os.path.exists(PKL_PATH):
    print(f"Cached interactome found at {PKL_PATH} — loading.")
    with open(PKL_PATH, "rb") as f:
        G = pickle.load(f)
else:
    print(f"Converting {CX_PATH} → {PKL_PATH} ...")

    if not os.path.exists(CX_PATH):
        raise FileNotFoundError(
            f"{CX_PATH} not found. "
            "Download PCNet 2.2 from NDEx and place it in data/input/ppi/."
        )

    with open(CX_PATH, "r") as f:
        cx_data = json.load(f)

    G = nx.Graph()

    # Add nodes (Entrez ID as node key, gene symbol as attribute)
    for entry in cx_data:
        if "nodes" in entry:
            for node in entry["nodes"]:
                G.add_node(int(node["@id"]), gene_symbol=node["n"])
            break

    # Add edges
    for entry in cx_data:
        if "edges" in entry:
            for edge in entry["edges"]:
                G.add_edge(int(edge["s"]), int(edge["t"]))
            break

    # Clean up: remove self-loops, keep largest connected component
    G.remove_edges_from(nx.selfloop_edges(G))
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()

    with open(PKL_PATH, "wb") as f:
        pickle.dump(G, f)
    print(f"Saved to {PKL_PATH}")

print(f"\nNodes        : {G.number_of_nodes():,}")
print(f"Edges        : {G.number_of_edges():,}")
print(f"Is connected : {nx.is_connected(G)}")

sample_nodes = list(G.nodes(data=True))[:5]
print("\nSample nodes (Entrez ID → attributes):")
for node_id, attrs in sample_nodes:
    print(f"  {node_id} → {attrs}")


Cached interactome found at data/processed/ppi/ppi_network.pkl — loading.

Nodes        : 18,554
Edges        : 3,323,926
Is connected : True

Sample nodes (Entrez ID → attributes):
  2 → {'gene_symbol': 'A2M'}
  351 → {'gene_symbol': 'APP'}
  4035 → {'gene_symbol': 'LRP1'}
  335 → {'gene_symbol': 'APOA1'}
  3240 → {'gene_symbol': 'HP'}


## Step 2 — Load MONDO and Parse Cross-Reference Tables

Loads the MONDO ontology, filters to non-obsolete `MONDO:` terms (excluding
`HP:` phenotype and other non-disease prefixes), and builds three lookup
dictionaries used for mapping in Steps 4 and 5:

| Dictionary | Key | Use |
|---|---|---|
| `omim_to_mondo` | OMIM MIM ID | Direct cross-reference (highest priority) |
| `name_to_mondo` | Lowercase disease name | Exact-name fallback |
| `synonym_to_mondo` | Lowercase synonym | Synonym fallback |

**Output:** `data/processed/disease/mondo_xref_table.csv`


In [3]:
import pronto
import pandas as pd

MONDO_PATH = "data/input/disease/mondo.obo"

print("Loading MONDO ontology (1–2 min on first run)...")
mondo = pronto.Ontology(MONDO_PATH)
print("MONDO loaded.")


Loading MONDO ontology (1–2 min on first run)...


/tmp/ipykernel_3966480/3069666421.py:7: UnicodeWarning: unsound encoding, assuming utf-8 (99% confidence)
  mondo = pronto.Ontology(MONDO_PATH)


MONDO loaded.


In [4]:
# ── Parse MONDO terms ────────────────────────────────────────────────────────
records = []

for term in mondo.terms():
    # Skip obsolete terms (handle both pronto API variants)
    try:
        is_obs = term.obsolete
    except AttributeError:
        try:
            is_obs = term.is_obsolete
        except AttributeError:
            is_obs = False
    if is_obs:
        continue

    # Keep only MONDO: disease concepts (HP: = phenotypes/symptoms → exclude)
    if not term.id.startswith("MONDO:"):
        continue

    mondo_id   = term.id
    mondo_name = term.name
    if not mondo_name:
        continue

    omim_ids, mesh_ids, efo_ids, doid_ids = [], [], [], []
    try:
        for xref in term.xrefs:
            xid = xref.id
            if   xid.startswith("OMIM:"): omim_ids.append(xid.replace("OMIM:", ""))
            elif xid.startswith("MESH:"): mesh_ids.append(xid.replace("MESH:", ""))
            elif xid.startswith("EFO:"):  efo_ids.append( xid.replace("EFO:", ""))
            elif xid.startswith("DOID:"): doid_ids.append(xid.replace("DOID:", ""))
    except Exception:
        pass

    try:
        syns = [s.description for s in term.synonyms]
    except Exception:
        syns = []

    records.append({
        "mondo_id":   mondo_id,
        "mondo_name": mondo_name,
        "omim_ids":   "|".join(omim_ids),
        "mesh_ids":   "|".join(mesh_ids),
        "mesh_id":    mesh_ids[0] if mesh_ids else "",
        "efo_ids":    "|".join(efo_ids),
        "doid_ids":   "|".join(doid_ids),
        "synonyms":   "|".join(syns),
    })

mondo_df = pd.DataFrame(records)
mondo_df.to_csv("data/processed/disease/mondo_xref_table.csv", index=False)

print(f"MONDO terms parsed       : {len(mondo_df):,}")
print(f"Terms with OMIM xref     : {(mondo_df['omim_ids'] != '').sum():,}")
print(f"Terms with MeSH xref     : {(mondo_df['mesh_ids'] != '').sum():,}")
print(f"Saved: data/processed/disease/mondo_xref_table.csv")

# ── Build lookup dictionaries ─────────────────────────────────────────────────
omim_to_mondo    = {}
name_to_mondo    = {}
synonym_to_mondo = {}

for _, row in mondo_df.iterrows():
    entry = {
        "mondo_id":   row["mondo_id"],
        "mondo_name": row["mondo_name"],
        "mesh_id":    row["mesh_id"],
    }
    if row["omim_ids"]:
        for oid in row["omim_ids"].split("|"):
            omim_to_mondo[oid.strip()] = entry
    if row["mondo_name"]:
        name_to_mondo[row["mondo_name"].lower()] = entry
    if row["synonyms"]:
        for syn in row["synonyms"].split("|"):
            syn = syn.strip().lower()
            if syn:
                synonym_to_mondo[syn] = entry

print(f"\nOMIM ID mappings   : {len(omim_to_mondo):,}")
print(f"Name mappings      : {len(name_to_mondo):,}")
print(f"Synonym mappings   : {len(synonym_to_mondo):,}")


MONDO terms parsed       : 26,660
Terms with OMIM xref     : 9,794
Terms with MeSH xref     : 8,125
Saved: data/processed/disease/mondo_xref_table.csv

OMIM ID mappings   : 9,887
Name mappings      : 26,660
Synonym mappings   : 105,192


In [5]:
# ── PART 2: Build MONDO ancestor map (depth-floor ceiling) ──────────────────────
#
# MONDO is structured hierarchically. When a gene is mapped to a specific disease
# (e.g. "lung adenocarcinoma"), we propagate that gene association upward to
# all ancestor terms (e.g. "lung cancer", "neoplasm by site", "neoplasm") up to a
# depth ceiling. This mirrors the MeSH upward propagation used in:
#   Menche et al. 2015 (Nat Methods) Supp. Fig. S2
#
# Depth ceiling rule: only propagate to ancestors whose BFS depth from the root
# MONDO:0000001 ("disease or disorder") is >= MONDO_DEPTH_FLOOR.
#
#   depth 0 = MONDO:0000001  "disease or disorder"          EXCLUDED
#   depth 1 = MONDO:0700096  "human disease"                EXCLUDED
#             MONDO:0003847  "hereditary disease"           EXCLUDED
#   depth 2 = MONDO:0005550  "infectious disease"           EXCLUDED at floor=3
#             MONDO:0004995  "cardiovascular disease"       EXCLUDED at floor=3
#   depth 3+ = leukemia, lung neoplasm, type 2 diabetes     INCLUDED
#
# Calibration: MONDO_DEPTH_FLOOR = 3 is the default value. OMIM-mapped terms
# (Step 2) all sit at depth >= 3, making this threshold safe. Structural
# groupers that survive this floor (e.g. "bone marrow disorder") are removed
# later by the direct-evidence filter (Step 5).

MONDO_DEPTH_FLOOR = 3   # terms with depth < this are excluded from ancestor sets

# ── Step 2a: Build children index and compute depth via BFS from root ─────────
from collections import deque, defaultdict

ROOT_ID = "MONDO:0000001"   # "disease or disorder"

# Single-hop is_a parents for every term
mondo_direct_parents = {}
for term in mondo.terms():
    try:
        is_obs = term.obsolete
    except AttributeError:
        try:
            is_obs = term.is_obsolete
        except AttributeError:
            is_obs = False
    if is_obs or not term.id.startswith("MONDO:"):
        continue
    try:
        parents = {
            p.id for p in term.superclasses(distance=1, with_self=False)
            if p.id.startswith("MONDO:")
        }
    except Exception:
        parents = set()
    mondo_direct_parents[term.id] = parents

# Reverse index: parent \u2192 set of direct children
children_of = defaultdict(set)
for tid, parents in mondo_direct_parents.items():
    for p in parents:
        children_of[p].add(tid)

# BFS from root \u2014 assigns shortest-path depth to every reachable term
mondo_depth = {}
queue = deque([(ROOT_ID, 0)])
visited = set()
while queue:
    node, d = queue.popleft()
    if node in visited:
        continue
    visited.add(node)
    mondo_depth[node] = d
    for child in children_of.get(node, set()):
        if child not in visited:
            queue.append((child, d + 1))

# Any term not reachable from the root (disconnected subgraph) gets max+1
max_known_depth = max(mondo_depth.values(), default=0)
for term in mondo.terms():
    if term.id.startswith("MONDO:") and term.id not in mondo_depth:
        mondo_depth[term.id] = max_known_depth + 1

print(f"MONDO depth computed for  : {len(mondo_depth)} terms")
print(f"Max depth in hierarchy    : {max(mondo_depth.values())}")

# ── Depth distribution summary ────────────────────────────────────────────────
from collections import Counter
depth_counts = Counter(mondo_depth.values())
print("\\nDepth distribution (levels 0\u201310):")
for d in sorted(depth_counts)[:11]:
    bar = "\u2588" * min(depth_counts[d] // 100, 60)
    print(f"  depth {d:2d} : {depth_counts[d]:6d} terms  {bar}")

# ── Show exactly which terms will be excluded (depth < MONDO_DEPTH_FLOOR) ─────
excluded_ids = {tid for tid, d in mondo_depth.items() if d < MONDO_DEPTH_FLOOR}
print(f"\\nTerms excluded by depth floor (depth < {MONDO_DEPTH_FLOOR}):")
for tid in sorted(excluded_ids):
    tname = mondo_df[mondo_df["mondo_id"] == tid]["mondo_name"].values
    tname = tname[0] if len(tname) else "?"
    print(f"  depth {mondo_depth[tid]}  {tid}  {tname}")

# Save depth table for inspection
depth_df = mondo_df[["mondo_id", "mondo_name"]].copy()
depth_df["depth"] = depth_df["mondo_id"].map(mondo_depth)
depth_df = depth_df.sort_values("depth")
depth_df.to_csv("data/processed/disease/mondo_depth_table.csv", index=False)
print(f"\\nSaved: data/processed/disease/mondo_depth_table.csv")

# ── Step 2b: Build ancestor map ──────────────────────────────────────────────
print("\\nBuilding MONDO ancestor map (depth-floor ceiling)...")
mondo_parents = {}

for term in mondo.terms():
    try:
        is_obs = term.obsolete
    except AttributeError:
        try:
            is_obs = term.is_obsolete
        except AttributeError:
            is_obs = False
    if is_obs or not term.id.startswith("MONDO:"):
        continue

    ancestors = set()
    try:
        for anc in term.superclasses(distance=None, with_self=False):
            anc_id = anc.id
            if (anc_id.startswith("MONDO:")
                    and mondo_depth.get(anc_id, 0) >= MONDO_DEPTH_FLOOR):
                ancestors.add(anc_id)
    except Exception:
        pass
    mondo_parents[term.id] = ancestors

n_with_parents = sum(1 for v in mondo_parents.values() if v)
print(f"MONDO terms indexed              : {len(mondo_parents)}")
print(f"Terms with \u22651 bounded ancestor   : {n_with_parents}")
avg_depth = sum(len(v) for v in mondo_parents.values()) / max(len(mondo_parents), 1)
print(f"Mean ancestor set size           : {avg_depth:.1f}")

# ── Root self-check: excluded terms should have empty ancestor sets ────────────
print("\\nDepth-floor self-check (excluded terms should be absent from all ancestor sets):")
leaked = set()
for term_id, ancs in mondo_parents.items():
    leaked |= (ancs & excluded_ids)
if leaked:
    leaked_names = [
        mondo_df[mondo_df["mondo_id"] == t]["mondo_name"].values
        for t in leaked
    ]
    leaked_names = [n[0] if len(n) else t for n, t in zip(leaked_names, leaked)]
    print(f"  \u26a0\ufe0f  {len(leaked)} excluded terms still appear in ancestor sets: {leaked_names}")
else:
    print(f"  \u2705 No excluded (depth < {MONDO_DEPTH_FLOOR}) terms found in any ancestor set")
    
print("\\n✅ MONDO depth hierarchy ready for ancestor propagation")


MONDO depth computed for  : 30538 terms
Max depth in hierarchy    : 12
\nDepth distribution (levels 0–10):
  depth  0 :      1 terms  
  depth  1 :      2 terms  
  depth  2 :    100 terms  █
  depth  3 :   4965 terms  █████████████████████████████████████████████████
  depth  4 :   7352 terms  ████████████████████████████████████████████████████████████
  depth  5 :   6042 terms  ████████████████████████████████████████████████████████████
  depth  6 :   4440 terms  ████████████████████████████████████████████
  depth  7 :   2077 terms  ████████████████████
  depth  8 :    844 terms  ████████
  depth  9 :    251 terms  ██
  depth 10 :     32 terms  
\nTerms excluded by depth floor (depth < 3):
  depth 0  MONDO:0000001  disease
  depth 2  MONDO:0002022  disorder of orbital region
  depth 2  MONDO:0002025  psychiatric disorder
  depth 2  MONDO:0002051  integumentary system disorder
  depth 2  MONDO:0002081  musculoskeletal system disorder
  depth 2  MONDO:0002118  urinary system disorde

## Step 3 — Verify Lookup Dictionaries

All MONDO IDs must carry a `MONDO:` prefix. The HP: contamination count must
be 0 in all three dictionaries before proceeding.


In [6]:
# ── Spot-check known OMIM IDs ─────────────────────────────────────────────────
test_omim_ids = ["222100", "104300", "114480"]
print("=== OMIM ID Lookups ===")
for oid in test_omim_ids:
    r  = omim_to_mondo.get(oid, "NOT FOUND")
    ok = isinstance(r, dict) and r["mondo_id"].startswith("MONDO:")
    print(f"  {'✅' if ok else '❌'} OMIM {oid} → {r}")

# ── Spot-check disease names ───────────────────────────────────────────────────
test_names = ["diabetes mellitus", "alzheimer disease", "breast cancer"]
print("\n=== Name Lookups ===")
for name in test_names:
    r  = name_to_mondo.get(name, "NOT FOUND")
    ok = isinstance(r, dict) and r["mondo_id"].startswith("MONDO:")
    print(f"  {'✅' if ok else '❌'} '{name}' → {r}")

# ── HP: contamination check ───────────────────────────────────────────────────
hp_names = sum(1 for v in name_to_mondo.values()    if v["mondo_id"].startswith("HP:"))
hp_omim  = sum(1 for v in omim_to_mondo.values()    if v["mondo_id"].startswith("HP:"))
hp_syns  = sum(1 for v in synonym_to_mondo.values() if v["mondo_id"].startswith("HP:"))
print("\n=== HP: Contamination Check (all must be 0) ===")
print(f"  name_to_mondo    : {hp_names}  {'✅' if hp_names == 0 else '❌'}")
print(f"  omim_to_mondo    : {hp_omim}   {'✅' if hp_omim  == 0 else '❌'}")
print(f"  synonym_to_mondo : {hp_syns}   {'✅' if hp_syns  == 0 else '❌'}")
if hp_names + hp_omim + hp_syns > 0:
    raise RuntimeError("HP: contamination detected — review MONDO filtering in Step 2.")


=== OMIM ID Lookups ===
  ✅ OMIM 222100 → {'mondo_id': 'MONDO:0005147', 'mondo_name': 'type 1 diabetes mellitus', 'mesh_id': 'D003922'}
  ✅ OMIM 104300 → {'mondo_id': 'MONDO:0007088', 'mondo_name': 'Alzheimer disease type 1', 'mesh_id': 'C536594'}
  ✅ OMIM 114480 → {'mondo_id': 'MONDO:0016419', 'mondo_name': 'hereditary breast carcinoma', 'mesh_id': 'C562840'}

=== Name Lookups ===
  ✅ 'diabetes mellitus' → {'mondo_id': 'MONDO:0005015', 'mondo_name': 'diabetes mellitus', 'mesh_id': 'D003920'}
  ✅ 'alzheimer disease' → {'mondo_id': 'MONDO:0004975', 'mondo_name': 'Alzheimer disease', 'mesh_id': 'D000544'}
  ✅ 'breast cancer' → {'mondo_id': 'MONDO:0007254', 'mondo_name': 'breast cancer', 'mesh_id': ''}

=== HP: Contamination Check (all must be 0) ===
  name_to_mondo    : 0  ✅
  omim_to_mondo    : 0   ✅
  synonym_to_mondo : 0   ✅


## Step 4 — Map OMIM Entries to MONDO

Maps `genemap2.txt` phenotypes to MONDO using four strategies in priority order:

| Priority | Strategy | Notes |
|---|---|---|
| 1 | OMIM MIM ID | Direct cross-reference — most reliable |
| 2 | Exact name match | Case-insensitive |
| 3 | Synonym match | All known MONDO synonyms |
| 4 | Conservative normalisation | Apostrophes + unambiguous abbreviations only |

Download `genemap2.txt` from https://omim.org/downloads and place it in
`data/input/disease/` before running this step.


In [7]:
import re

def normalize_trait(text: str) -> str:
    """Conservative normalisation — apostrophes + unambiguous abbreviations only.

    Does NOT strip qualifiers (late-onset, chronic, etc.) to avoid merging
    biologically distinct disease subtypes.
    """
    text = text.lower().strip()
    text = text.replace("'s ", " ").replace("'s", "").replace("'", "")
    unambiguous = {
        r"\bt2d\b":  "type 2 diabetes mellitus",
        r"\bt1d\b":  "type 1 diabetes mellitus",
        r"\bibd\b":  "inflammatory bowel disease",
        r"\bsle\b":  "systemic lupus erythematosus",
        r"\bcopd\b": "chronic obstructive pulmonary disease",
        r"\bcad\b":  "coronary artery disease",
        r"\bckd\b":  "chronic kidney disease",
        r"\bra\b":   "rheumatoid arthritis",
    }
    for pat, rep in unambiguous.items():
        text = re.sub(pat, rep, text)
    return re.sub(r"\s+", " ", text).strip()


# ── Load genemap2.txt ─────────────────────────────────────────────────────────
GENEMAP_PATH = "data/input/disease/genemap2.txt"

if not os.path.exists(GENEMAP_PATH):
    raise FileNotFoundError(
        f"{GENEMAP_PATH} not found. Download from https://omim.org/downloads."
    )

omim = pd.read_csv(GENEMAP_PATH, sep="\t", comment="#", header=None, low_memory=False)
omim.columns = [
    "chromosome", "genomic_pos_start", "genomic_pos_end",
    "cyto_location", "computed_cyto_location", "mim_number",
    "gene_symbols", "gene_name", "approved_symbol",
    "entrez_gene_id", "ensembl_gene_id", "comments",
    "phenotypes", "mouse_gene_symbol",
]
omim = omim[omim["entrez_gene_id"].notna() & omim["phenotypes"].notna()].copy()
omim["entrez_gene_id"] = omim["entrez_gene_id"].astype(int)
print(f"OMIM entries loaded: {len(omim):,}")

# ── Map phenotypes → MONDO ────────────────────────────────────────────────────
omim_mapped = []
omim_failed = []

for _, row in omim.iterrows():
    for pheno in str(row["phenotypes"]).split(";"):
        pc = pheno.strip().split(",")[0]
        pc = pc.replace("{","").replace("}","").replace("[","").replace("]","")
        pc = pc.replace("?","").strip()
        if len(pc) < 4:
            continue

        # Extract MIM number if present (6-digit trailing token)
        mim_match = None
        for part in pheno.split(","):
            p = part.strip()
            if p.isdigit() and len(p) == 6:
                mim_match = p
                break

        mapping = match_method = None

        if mim_match and mim_match in omim_to_mondo:
            mapping, match_method = omim_to_mondo[mim_match].copy(), "omim_id"
        elif pc.lower() in name_to_mondo:
            mapping, match_method = name_to_mondo[pc.lower()].copy(), "exact_name"
        elif pc.lower() in synonym_to_mondo:
            mapping, match_method = synonym_to_mondo[pc.lower()].copy(), "synonym"
        else:
            norm = normalize_trait(pc)
            if norm in name_to_mondo:
                mapping, match_method = name_to_mondo[norm].copy(), "normalized_name"
            elif norm in synonym_to_mondo:
                mapping, match_method = synonym_to_mondo[norm].copy(), "normalized_synonym"

        if mapping:
            omim_mapped.append({
                "original_term":  pc,
                "entrez_gene_id": int(row["entrez_gene_id"]),
                "mondo_id":       mapping["mondo_id"],
                "mondo_name":     mapping["mondo_name"],
                "mesh_id":        mapping["mesh_id"],
                "match_method":   match_method,
                "source":         "OMIM",
            })
        else:
            omim_failed.append(pc)

omim_mapped_df = pd.DataFrame(omim_mapped)
omim_failed_counts = (
    pd.Series(omim_failed).value_counts()
    .rename_axis("failed_term").reset_index(name="frequency")
)

print(f"\nOMIM entries mapped      : {len(omim_mapped_df):,}")
print(f"OMIM unique terms failed : {len(omim_failed_counts):,}")
print(f"\nMatch method breakdown:")
print(omim_mapped_df["match_method"].value_counts().to_string())
print(f"\nTop 10 failed OMIM terms:")
print(omim_failed_counts.head(10).to_string(index=False))


OMIM entries loaded: 5,532

OMIM entries mapped      : 6,488
OMIM unique terms failed : 969

Match method breakdown:
match_method
exact_name    4357
synonym       2131

Top 10 failed OMIM terms:
                                                                    failed_term  frequency
                                            Intellectual developmental disorder         65
                                                             Ciliary dyskinesia         46
                                                                    Blood group         37
                                                            Myasthenic syndrome         31
                                                                  Short stature         15
                                                          Ceroid lipofuscinosis         13
                                                                       Aneurysm         12
Muscular dystrophy-dystroglycanopathy (congenital with brain and eye anomalie

## Step 5 — Map GWAS Catalog Entries to MONDO

Download the full associations file from
https://www.ebi.ac.uk/gwas/docs/file-downloads and place it in
`data/input/disease/gwas_catalog_associations.tsv` before running.

Applies genome-wide significance threshold (p ≤ 5 × 10⁻⁸) and converts
gene symbols to Entrez IDs via the local NCBI `gene_info` file. Mapping strategies mirror Step 4
(excluding the MIM-ID route, which is OMIM-specific).


In [8]:
from utils import build_symbol_to_entrez  # local NCBI gene_info — no HTTP

GWAS_PATH = "data/input/disease/gwas_catalog_associations.tsv"

if not os.path.exists(GWAS_PATH):
    raise FileNotFoundError(
        f"{GWAS_PATH} not found. Download from https://www.ebi.ac.uk/gwas/docs/file-downloads."
    )

gwas = pd.read_csv(GWAS_PATH, sep="\t", low_memory=False)
gwas["P-VALUE"] = pd.to_numeric(gwas["P-VALUE"], errors="coerce")
gwas = gwas[gwas["P-VALUE"] <= 5e-8].copy()
gwas = gwas[gwas["MAPPED_GENE"].notna()].copy()

print(f"GWAS associations (p ≤ 5e-8) : {len(gwas):,}")
print(f"Unique traits                 : {gwas['DISEASE/TRAIT'].nunique():,}")

# ── Symbol → Entrez conversion (local NCBI gene_info — reproducible) ─────────
all_symbols: set = set()
for entry in gwas["MAPPED_GENE"].dropna():
    for sym in entry.replace(" - ", ", ").replace(";", ",").split(","):
        sym = sym.strip()
        if sym:
            all_symbols.add(sym)

print(f"\nUnique gene symbols to convert: {len(all_symbols):,}")
symbol_to_entrez: dict = build_symbol_to_entrez(all_symbols)
print(f"Successfully converted        : {len(symbol_to_entrez):,} symbols")

# ── Map GWAS traits → MONDO ───────────────────────────────────────────────────
gwas_mapped = []
gwas_failed = []

for _, row in gwas.iterrows():
    trait = str(row["DISEASE/TRAIT"]).strip()
    mapping = match_method = None

    if trait.lower() in name_to_mondo:
        mapping, match_method = name_to_mondo[trait.lower()].copy(), "exact_name"
    elif trait.lower() in synonym_to_mondo:
        mapping, match_method = synonym_to_mondo[trait.lower()].copy(), "synonym"
    else:
        norm = normalize_trait(trait)
        if norm in name_to_mondo:
            mapping, match_method = name_to_mondo[norm].copy(), "normalized_name"
        elif norm in synonym_to_mondo:
            mapping, match_method = synonym_to_mondo[norm].copy(), "normalized_synonym"

    if not mapping:
        gwas_failed.append(trait)
        continue

    for sym in str(row["MAPPED_GENE"]).replace(" - ", ", ").replace(";", ",").split(","):
        sym = sym.strip()
        if sym in symbol_to_entrez:
            gwas_mapped.append({
                "original_term":  trait,
                "entrez_gene_id": symbol_to_entrez[sym],
                "mondo_id":       mapping["mondo_id"],
                "mondo_name":     mapping["mondo_name"],
                "mesh_id":        mapping["mesh_id"],
                "match_method":   match_method,
                "source":         "GWAS",
            })

gwas_mapped_df = pd.DataFrame(gwas_mapped)
gwas_failed_counts = (
    pd.Series(gwas_failed).value_counts()
    .rename_axis("failed_term").reset_index(name="frequency")
)

print(f"\nGWAS entries mapped      : {len(gwas_mapped_df):,}")
print(f"GWAS unique traits failed: {len(gwas_failed_counts):,}")
print(f"\nMatch method breakdown:")
print(gwas_mapped_df["match_method"].value_counts().to_string())
print(f"\nTop 10 failed GWAS traits:")
print(gwas_failed_counts.head(10).to_string(index=False))


GWAS associations (p ≤ 5e-8) : 755,880
Unique traits                 : 36,550

Unique gene symbols to convert: 30,616
Successfully converted        : 28,172 symbols

GWAS entries mapped      : 79,448
GWAS unique traits failed: 36,019

Match method breakdown:
match_method
exact_name            54404
synonym               24896
normalized_name         114
normalized_synonym       34

Top 10 failed GWAS traits:
                                                    failed_term  frequency
                                                         Height      23367
                                                Body mass index      12950
                                              Height (baseline)       6906
                                        Systolic blood pressure       5458
                                      Bone mineral density mean       5309
Electrocardiogram morphology (amplitude at temporal datapoints)       5261
                                                 Platelet count

## Step 6 — Combine, Filter, and Save Disease Gene Sets

Merges OMIM and GWAS gene sets, filters to diseases with ≥ 20 genes present
in the PCNet 2.2 interactome (threshold from Menche et al. 2015), and saves
the intermediate `gene_modules.pkl`.

> The spaceflight module (Step 7) is injected into this file after it is built.
> The final pickle will contain both disease gene sets **and** the spaceflight
> query module under the key `"spaceflight"`.

**Outputs:**
- `data/processed/gene_modules/gene_modules.pkl`  *(spaceflight added in Step 7)*
- `data/output/csv/disease_gene_summary.csv`
- `data/output/csv/disease_gene_mapping_audit.csv`
- `data/output/csv/failed_mappings_for_review.csv`
- `data/output/csv/removed_diseases.csv`


In [9]:
import pickle

PKL_PATH    = "data/processed/ppi/ppi_network.pkl"
DISEASE_OUT = "data/processed/gene_modules/gene_modules.pkl"

# Load interactome if not already in memory
if "G" not in vars():
    with open(PKL_PATH, "rb") as f:
        G = pickle.load(f)
ppi_nodes = set(G.nodes())
print(f"Interactome nodes: {len(ppi_nodes):,}")

all_mapped = pd.concat([omim_mapped_df, gwas_mapped_df], ignore_index=True)
print(f"Total mapped entries : {len(all_mapped):,}")
print(f"  From OMIM          : {len(omim_mapped_df):,}")
print(f"  From GWAS          : {len(gwas_mapped_df):,}")

omim_diseases = set(omim_mapped_df["mondo_name"].unique())
gwas_diseases = set(gwas_mapped_df["mondo_name"].unique())
print(f"\\nOMIM-only diseases   : {len(omim_diseases - gwas_diseases):,}")
print(f"GWAS-only diseases   : {len(gwas_diseases - omim_diseases):,}")
print(f"In both sources      : {len(omim_diseases & gwas_diseases):,}")
print(f"Total unique         : {len(omim_diseases | gwas_diseases):,}")

# ── STEP 5a: Build flat gene sets per MONDO concept (direct evidence only) ─
# Collect genes DIRECTLY mapped to each MONDO ID (no propagation yet).
# This set is used both for propagation and as the direct-evidence filter.
print("\\nBuilding direct gene sets...")
direct_genes = {}   # mondo_id -> set of entrez gene IDs (direct evidence only)
for mondo_id, grp in all_mapped.groupby("mondo_id"):
    genes = set(grp["entrez_gene_id"].dropna().astype(int).tolist())
    if genes:
        direct_genes[mondo_id] = genes

print(f"MONDO concepts with direct gene associations : {len(direct_genes)}")

# ── STEP 5b: Propagate genes upward through the MONDO hierarchy ──────────────
# For each MONDO concept that has direct gene associations, add those genes
# to every ancestor concept up to the depth ceiling set in Step 2.
# This mirrors the MeSH upward propagation in Menche et al. 2015 Supp. Fig. S2.
print("Propagating gene associations up the MONDO hierarchy...")

propagated_genes = {}   # mondo_id -> set of entrez gene IDs (direct + inherited)

# Seed with direct associations
for mondo_id, genes in direct_genes.items():
    propagated_genes[mondo_id] = set(genes)

# Walk every direct-mapped term and push its genes to all bounded ancestors
n_propagation_events = 0
for child_id, genes in direct_genes.items():
    for ancestor_id in mondo_parents.get(child_id, set()):
        if ancestor_id not in propagated_genes:
            propagated_genes[ancestor_id] = set()
        before = len(propagated_genes[ancestor_id])
        propagated_genes[ancestor_id].update(genes)
        n_propagation_events += len(propagated_genes[ancestor_id]) - before

print(f"MONDO concepts after propagation             : {len(propagated_genes)}")
print(f"Total gene-concept associations added        : {n_propagation_events:,}")

# ── STEP 5c: Build mondo_id \u2192 mondo_name lookup ──────────────────────────
id_to_name = dict(zip(mondo_df["mondo_id"], mondo_df["mondo_name"]))

# ── STEP 5d: Filter ────────────────────────────────────────────────────────
# Two conditions must both be satisfied:
#   1. ≥ 20 genes present in the PPI interactome
#   2. ≥ 1 gene directly mapped from OMIM or GWAS  (new: direct-evidence filter)
#
# Condition 2 removes pure-propagation terms — MONDO grouper nodes whose gene
# sets come entirely from upward inheritance with no independent experimental
# support. This prevents spurious associations with very broad categories.
print("\\nApplying filters:")
print(f"  Filter 1 \u2014 \u226520 genes in PPI interactome")
print(f"  Filter 2 \u2014 \u226501 directly mapped gene (OMIM or GWAS)  [new: direct-evidence filter]")

disease_genes    = {}
removed_diseases = []

for mondo_id, genes in propagated_genes.items():
    mondo_name      = id_to_name.get(mondo_id, mondo_id)
    genes_in_ppi    = genes & ppi_nodes
    n_direct_in_ppi = len(direct_genes.get(mondo_id, set()) & ppi_nodes)

    passes_size   = len(genes_in_ppi) >= 20
    passes_direct = n_direct_in_ppi >= 1

    if passes_size and passes_direct:
        disease_genes[mondo_name] = genes_in_ppi
    else:
        if not passes_size and not passes_direct:
            reason = "below 20 genes in PPI; no direct evidence"
        elif not passes_size:
            reason = "no genes in PPI" if len(genes_in_ppi) == 0 else "below 20 genes in PPI"
        else:
            reason = "pure propagation \u2014 no direct OMIM/GWAS evidence"
        removed_diseases.append({
            "mondo_id":         mondo_id,
            "disease":          mondo_name,
            "raw_gene_count":   len(genes),
            "ppi_gene_count":   len(genes_in_ppi),
            "direct_gene_count": n_direct_in_ppi,
            "reason":           reason,
        })

n_removed_pure_prop = sum(
    1 for r in removed_diseases if r["reason"] == "pure propagation \u2014 no direct OMIM/GWAS evidence"
)
print(f"\\nDiseases retained                    : {len(disease_genes)}")
print(f"Diseases removed                     : {len(removed_diseases)}")
print(f"  of which pure-propagation groupers : {n_removed_pure_prop}")
if disease_genes:
    print(f"Total unique genes across all diseases: {len(set.union(*disease_genes.values()))}")

# ── Propagation impact summary ────────────────────────────────────────────────
# "direct_pass" = terms that would have \u226520 PPI genes even without propagation
direct_pass = sum(1 for mid, g in direct_genes.items() if len(g & ppi_nodes) >= 20)
prop_unlock = len(disease_genes) - direct_pass
print(f"\\nPropagation impact:")
print(f"  Terms passing \u226520 PPI filter from direct evidence alone : {direct_pass}")
print(f"  Additional terms unlocked by propagation                : {prop_unlock}")

# ── Save intermediate pickle ──────────────────────────────────────────────────
with open(DISEASE_OUT, "wb") as f:
    pickle.dump(disease_genes, f)
print(f"\\nSaved: {DISEASE_OUT}  (spaceflight module added in Step 7)")

# ── Audit CSVs ────────────────────────────────────────────────────────────────
summary_rows = []
for mondo_name, genes in disease_genes.items():
    mid_matches = mondo_df[mondo_df["mondo_name"] == mondo_name]["mondo_id"].values
    mid = mid_matches[0] if len(mid_matches) > 0 else ""

    direct_count = len(direct_genes.get(mid, set()) & ppi_nodes)
    total_count  = len(genes)

    summary_rows.append({
        "mondo_id":             mid,
        "mondo_name":           mondo_name,
        "mondo_depth":          mondo_depth.get(mid, -1),
        "n_genes_ppi":          total_count,
        "n_genes_direct":       direct_count,
        "n_genes_propagated":   total_count - direct_count,
        "passed_direct_filter": direct_count >= 1,
        "in_omim":              mondo_name in omim_diseases,
        "in_gwas":              mondo_name in gwas_diseases,
    })

summary_df = (pd.DataFrame(summary_rows)
              .sort_values("n_genes_ppi", ascending=False))
summary_df.to_csv("data/output/csv/disease_gene_summary.csv", index=False)

all_mapped.to_csv("data/output/csv/disease_gene_mapping_audit.csv", index=False)

omim_failed_counts["failed_source"] = "OMIM"
gwas_failed_counts["failed_source"]  = "GWAS"
all_failed = (
    pd.concat([omim_failed_counts, gwas_failed_counts])
    .groupby(["failed_term", "failed_source"])["frequency"]
    .sum().reset_index()
    .sort_values("frequency", ascending=False)
)
all_failed.to_csv("data/output/csv/failed_mappings_for_review.csv", index=False)

_df_removed = pd.DataFrame(removed_diseases)
if not _df_removed.empty:
    _df_removed = _df_removed.sort_values("ppi_gene_count", ascending=False)
_df_removed.to_csv("data/output/csv/removed_diseases.csv", index=False)

print("Saved: data/output/csv/disease_gene_summary.csv")
print("Saved: data/output/csv/disease_gene_mapping_audit.csv")
print("Saved: data/output/csv/failed_mappings_for_review.csv")
print("Saved: data/output/csv/removed_diseases.csv")
print(f"\\nTop 10 diseases by gene count:")
print(summary_df.head(10).to_string(index=False))

# ── Show direct vs propagated gene counts ──────────────────────────────────────
print(f"\\nDirect-evidence statistics:")
print(f"  Diseases with 100% direct genes    : {(summary_df['n_genes_propagated'] == 0).sum()}")
print(f"  Diseases with mixed direct+prop    : {((summary_df['n_genes_propagated'] > 0) & (summary_df['n_genes_direct'] > 0)).sum()}")
print(f"  Diseases with only 1 direct gene   : {(summary_df['n_genes_direct'] == 1).sum()}")


Interactome nodes: 18,554
Total mapped entries : 85,936
  From OMIM          : 6,488
  From GWAS          : 79,448
\nOMIM-only diseases   : 3,634
GWAS-only diseases   : 334
In both sources      : 166
Total unique         : 4,134
\nBuilding direct gene sets...
MONDO concepts with direct gene associations : 4134
Propagating gene associations up the MONDO hierarchy...
MONDO concepts after propagation             : 6091
Total gene-concept associations added        : 167,670
\nApplying filters:
  Filter 1 — ≥20 genes in PPI interactome
  Filter 2 — ≥01 directly mapped gene (OMIM or GWAS)  [new: direct-evidence filter]
\nDiseases retained                    : 265
Diseases removed                     : 5826
  of which pure-propagation groupers : 421
Total unique genes across all diseases: 9225
\nPropagation impact:
  Terms passing ≥20 PPI filter from direct evidence alone : 167
  Additional terms unlocked by propagation                : 98
\nSaved: data/processed/gene_modules/gene_modules.pkl

## Step 7 — Build Spaceflight Module and Inject into Gene Modules

Compiles the spaceflight query module from three missions and injects it into
`gene_modules.pkl` as `"spaceflight"` — the reference module that all OMIM/GWAS
disease gene sets will be compared against in the separation pipeline.

### Dataset selection criteria

| Dataset | File | Comparison | Threshold | Rationale |
|---|---|---|---|---|
| JAXA CFE | `Supplementary_Data_1.xlsx` (Ohshima et al. 2022) | Pre vs In-flight | FDR < 0.05 (all 472) | Defines spaceflight — direct in-flight blood draw |
| NASA Twins | `aau8650_table_s2.xlsx` (Garrett-Bakelman et al. 2019) | In-flight vs Pre-flight, Multivariate only | adj p < 0.05 | TW self-comparison; Multivariate uses PolyA+ and Ribodepleted jointly |
| Inspiration4 | `Supplementary_Data_2.xlsx` (Malkani et al. 2023) | FP (R+1 vs Pre), DESeq2 OR edgeR | adj p < 0.05 | No in-flight draw; R+1 is closest post-flight timepoint for a 3-day mission |

### Sanity check against NASA Twins paper
- Paper reports 9,317 total DEGs (HR ground twin as control, q < 0.01) — this maps
  to the `In-flight, 2nd Half vs Ground` PBMC comparison, **not** our selection.
- Our selection (`In-flight vs Pre-flight`, Multivariate, q < 0.05) is the TW
  self-comparison that aligns with JAXA's Pre vs In-flight design.


In [10]:
import numpy as np

# ── Reload ppi_nodes if running standalone ────────────────────────────────────
if "ppi_nodes" not in vars():
    with open("data/processed/ppi/ppi_network.pkl", "rb") as f:
        G = pickle.load(f)
    ppi_nodes = set(G.nodes())
    print(f"Interactome reloaded: {len(ppi_nodes):,} nodes")

# ════════════════════════════════════════════════════════════════════════════
# SOURCE 1 — JAXA CFE  (Pre vs In-flight, FDR < 0.05, 472 DEGs)
# Ohshima et al. 2022, "Release of CD36-associated cell-free mitochondrial
# DNA and RNA as a hallmark of space environment response"
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("SOURCE 1: JAXA CFE")
print("=" * 60)

jaxa = pd.read_excel("data/input/spaceflight/Supplementary_Data_1.xlsx", sheet_name="Pre_vs_FL_472_genes")
jaxa.rename(columns={
    "Feature ID": "gene_symbol",
    "EDGE test: Pre vs Flight, tagwise dispersions - FDR p-value correction": "fdr_pre_vs_flight",
    "EDGE test: Pre vs Flight, tagwise dispersions - Fold change":             "fc_pre_vs_flight",
}, inplace=True)

jaxa_sig = jaxa[jaxa["fdr_pre_vs_flight"] < 0.05]
print(f"Total genes in sheet    : {len(jaxa)}")
print(f"Passing FDR < 0.05      : {len(jaxa_sig)}")
print(f"Upregulated (FC > 0)    : {(jaxa_sig['fc_pre_vs_flight'] > 0).sum()}")
print(f"Downregulated (FC < 0)  : {(jaxa_sig['fc_pre_vs_flight'] < 0).sum()}")

jaxa_symbols = jaxa_sig["gene_symbol"].dropna().astype(str).tolist()
_jaxa_map = build_symbol_to_entrez(jaxa_symbols)
jaxa_entrez = set(_jaxa_map.values())
jaxa_in_ppi = jaxa_entrez & ppi_nodes
print(f"Converted to Entrez IDs : {len(jaxa_entrez)}")
print(f"In interactome          : {len(jaxa_in_ppi)}")


SOURCE 1: JAXA CFE
Total genes in sheet    : 472
Passing FDR < 0.05      : 472
Upregulated (FC > 0)    : 15
Downregulated (FC < 0)  : 457
Converted to Entrez IDs : 389
In interactome          : 301


In [11]:
# ════════════════════════════════════════════════════════════════════════════
# SOURCE 2 — NASA Twins Study
# Garrett-Bakelman et al. 2019, "The NASA Twins Study: A multidimensional
# analysis of a year-long human spaceflight"
# Selection: In-flight vs Pre-flight | Multivariate | adj p < 0.05
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("SOURCE 2: NASA Twins Study")
print("=" * 60)

twins = pd.read_excel("data/input/spaceflight/aau8650_table_s2.xlsx",
                      sheet_name="All DEGs", header=1)
twins["adjusted p-value"] = pd.to_numeric(twins["adjusted p-value"], errors="coerce")

twins_selected = twins[
    (twins["Coefficient"]    == "In-flight vs Pre-flight") &
    (twins["ExperimentType"] == "Multivariate (PolyA+ and Ribodepleted together)") &
    (twins["adjusted p-value"] < 0.05)
].copy()

print(f"Rows selected (all cell types): {len(twins_selected)}")
print(f"Unique gene symbols           : {twins_selected['Gene'].nunique()}")
print(f"\nCell type breakdown:")
print(twins_selected.groupby("CellType")["Gene"].nunique().to_string())
print(f"\nDirection split:")
print(twins_selected["Dir"].value_counts().to_string())

twins_symbols = twins_selected["Gene"].dropna().unique().tolist()
_twins_map = build_symbol_to_entrez(twins_symbols)
twins_entrez = set(_twins_map.values())
twins_in_ppi = twins_entrez & ppi_nodes
print(f"\nConverted to Entrez IDs : {len(twins_entrez)}")
print(f"In interactome          : {len(twins_in_ppi)}")


SOURCE 2: NASA Twins Study
Rows selected (all cell types): 1122
Unique gene symbols           : 999

Cell type breakdown:
CellType
CD4    799
CD8    134
LD     189

Direction split:
Dir
Down-regulated    758
Up-regulated      364

Converted to Entrez IDs : 943
In interactome          : 826


In [12]:
# ════════════════════════════════════════════════════════════════════════════
# SOURCE 3 — Inspiration4
# Malkani et al. 2023, "Direct RNA sequencing of astronaut blood reveals
# spaceflight-associated m6A increases and hematopoietic transcriptional responses"
# Selection: FP (R+1 vs Pre), DESeq2 OR edgeR, adj p < 0.05
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("SOURCE 3: Inspiration4")
print("=" * 60)

i4_raw = pd.read_excel("data/input/spaceflight/Supplementary_Data_2.xlsx",
                       sheet_name="Supplemental-Table-S2(5)", header=None)

# Build column index from 4-row header structure
groups    = i4_raw.iloc[1, :].tolist()
pipelines = i4_raw.iloc[2, :].tolist()
stats_row = i4_raw.iloc[3, :].tolist()

fp_deseq2_adjp, fp_edger_adjp = [], []
for c in range(len(groups)):
    g, p, s = groups[c], pipelines[c], stats_row[c]
    if isinstance(g, str) and g.startswith("I4-FP"):
        if p == "DESeq2" and s == "adjusted p-value": fp_deseq2_adjp.append(c)
        if p == "edgeR"  and s == "adjusted p-value": fp_edger_adjp.append(c)

print(f"FP DESeq2 adj p-val columns : {fp_deseq2_adjp}")
print(f"FP edgeR  adj p-val columns : {fp_edger_adjp}")

data_i4 = i4_raw.iloc[4:, :].copy()
data_i4.columns = range(data_i4.shape[1])

deseq2_vals = data_i4[fp_deseq2_adjp].apply(pd.to_numeric, errors="coerce")
edger_vals  = data_i4[fp_edger_adjp ].apply(pd.to_numeric, errors="coerce")

sig_deseq2 = (deseq2_vals < 0.05).any(axis=1)
sig_edger  = (edger_vals  < 0.05).any(axis=1)
sig_either = sig_deseq2 | sig_edger
sig_both   = sig_deseq2 & sig_edger

print(f"\nSig in DESeq2 (any FP subject)  : {sig_deseq2.sum()}")
print(f"Sig in edgeR  (any FP subject)  : {sig_edger.sum()}")
print(f"Sig in EITHER (union — selected): {sig_either.sum()}")
print(f"Sig in BOTH   (intersection)    : {sig_both.sum()}")

i4_ensembl_raw = data_i4.loc[sig_either, 0].dropna().astype(str).tolist()
# Strip version suffix (e.g. ENSG00000000938.13 → ENSG00000000938)
i4_ensembl = [eid.split(".")[0] for eid in i4_ensembl_raw if eid.upper().startswith("ENSG")]
print(f"\nEnsembl IDs to convert          : {len(i4_ensembl)}")

from utils import build_ensembl_to_entrez
_i4_map = build_ensembl_to_entrez(i4_ensembl)
i4_entrez = set(_i4_map.values())
i4_in_ppi = i4_entrez & ppi_nodes
print(f"Converted to Entrez IDs         : {len(i4_entrez)}")
print(f"In interactome                  : {len(i4_in_ppi)}")


SOURCE 3: Inspiration4
FP DESeq2 adj p-val columns : [59, 65, 71]
FP edgeR  adj p-val columns : [62, 68, 74]

Sig in DESeq2 (any FP subject)  : 16
Sig in edgeR  (any FP subject)  : 75
Sig in EITHER (union — selected): 81
Sig in BOTH   (intersection)    : 10

Ensembl IDs to convert          : 81
Converted to Entrez IDs         : 66
In interactome                  : 60


In [13]:
# ════════════════════════════════════════════════════════════════════════════
# COMBINE — Union of all three sources
# ════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("SPACEFLIGHT MODULE — COMBINED")
print("=" * 60)

spaceflight_genes = jaxa_in_ppi | twins_in_ppi | i4_in_ppi

print(f"JAXA CFE genes in PPI     : {len(jaxa_in_ppi)}")
print(f"NASA Twins genes in PPI   : {len(twins_in_ppi)}")
print(f"Inspiration4 genes in PPI : {len(i4_in_ppi)}")
print()
print(f"JAXA ∩ Twins              : {len(jaxa_in_ppi & twins_in_ppi)}")
print(f"JAXA ∩ I4                 : {len(jaxa_in_ppi & i4_in_ppi)}")
print(f"Twins ∩ I4                : {len(twins_in_ppi & i4_in_ppi)}")
print(f"All three sources (core)  : {len(jaxa_in_ppi & twins_in_ppi & i4_in_ppi)}")
print()
print(f"Total union (final module): {len(spaceflight_genes)}")

if len(spaceflight_genes) < 20:
    print(f"\n⚠️  WARNING: Only {len(spaceflight_genes)} genes — below 20-gene minimum.")
    print("   Separation metrics may be unreliable.")
else:
    print("\n✅ Gene count sufficient for network separation analysis.")

# ── Per-source audit table ────────────────────────────────────────────────────
spaceflight_audit = pd.DataFrame([
    {
        "entrez_gene_id": g,
        "in_jaxa":   g in jaxa_in_ppi,
        "in_twins":  g in twins_in_ppi,
        "in_i4":     g in i4_in_ppi,
        "n_sources": sum([g in jaxa_in_ppi, g in twins_in_ppi, g in i4_in_ppi]),
    }
    for g in spaceflight_genes
]).sort_values("n_sources", ascending=False)

spaceflight_audit.to_csv("data/output/csv/spaceflight_gene_audit.csv", index=False)
print(f"\nSaved: data/output/csv/spaceflight_gene_audit.csv")
print(f"\nGenes by source count:")
print(spaceflight_audit["n_sources"].value_counts().sort_index(ascending=False).to_string())


SPACEFLIGHT MODULE — COMBINED
JAXA CFE genes in PPI     : 301
NASA Twins genes in PPI   : 826
Inspiration4 genes in PPI : 60

JAXA ∩ Twins              : 9
JAXA ∩ I4                 : 0
Twins ∩ I4                : 12
All three sources (core)  : 0

Total union (final module): 1166

✅ Gene count sufficient for network separation analysis.

Saved: data/output/csv/spaceflight_gene_audit.csv

Genes by source count:
n_sources
2      21
1    1145


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# INJECT spaceflight module → disease_genes and resave
# ════════════════════════════════════════════════════════════════════════════
SPACEFLIGHT_KEY = "spaceflight"
DISEASE_OUT     = "data/processed/gene_modules/gene_modules.pkl"

# Reload disease_genes if running this block standalone
if "disease_genes" not in vars():
    with open(DISEASE_OUT, "rb") as f:
        disease_genes = pickle.load(f)
    print(f"disease_genes reloaded: {len(disease_genes)} sets")

print(f"Disease gene sets before injection : {len(disease_genes)}")

if SPACEFLIGHT_KEY in disease_genes:
    print(f"\n⚠️  '{SPACEFLIGHT_KEY}' already exists — overwriting.")
    print(f"   Old gene count: {len(disease_genes[SPACEFLIGHT_KEY])}")

disease_genes[SPACEFLIGHT_KEY] = spaceflight_genes

with open(DISEASE_OUT, "wb") as f:
    pickle.dump(disease_genes, f)

# ── Verify round-trip ─────────────────────────────────────────────────────────
with open(DISEASE_OUT, "rb") as f:
    verify = pickle.load(f)

status = "✅ CONFIRMED" if SPACEFLIGHT_KEY in verify else "❌ NOT FOUND"
print(f"\n{'=' * 50}")
print("SPACEFLIGHT MODULE INJECTED")
print(f"{'=' * 50}")
print(f"  Key               : '{SPACEFLIGHT_KEY}'")
print(f"  Genes in module   : {len(spaceflight_genes)}")
print(f"  Total disease sets: {len(disease_genes)}")
print(f"  Reload verify     : {status}")
print()
print(f"→ Proceed to  2_separation_pipeline.ipynb")
print(f"  Query module  : '{SPACEFLIGHT_KEY}'")
print(f"  Target diseases: {len(disease_genes) - 1}")


Disease gene sets before injection : 265

SPACEFLIGHT MODULE INJECTED
  Key               : 'spaceflight'
  Genes in module   : 1166
  Total disease sets: 266
  Reload verify     : ✅ CONFIRMED

→ Proceed to  2_separation_pipeline.ipynb
  Query module  : 'spaceflight'
  Target diseases: 265
